In [ ]:
!pip install --upgrade ipython jupyter
%load_ext autoreload
%autoreload 2


def refresh_repo():
    %cd /kaggle/working
    %rm -rf from-neurons-to-directions
    !git clone https://github.com/jefri021/from-neurons-to-directions.git
    %cd /kaggle/working/from-neurons-to-directions/
    !git pull origin main

refresh_repo()

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/from-neurons-to-directions/src")

import torch
import matplotlib.pyplot as plt

from model_utils import (
    load_model_and_tokenizer,
    apply_chat_template,
    get_num_layers,
    generate,
)
from activation_store import ActivationStore
from safety_neurons import (
    collect_generated_span_activations,
    compute_change_scores_rms,
    get_top_safety_neurons,
    compute_causal_effect,
    generate_with_neuron_ablation,
)
from metrics import refusal_rate
from viz import (
    plot_change_score_heatmap,
    plot_neuron_layer_dist,
    plot_causal_effect_bar,
    plot_refusal_rates,
)

print("Imports OK")

In [ ]:
# Try fitting base model back in
base_model, base_tokenizer = load_model_and_tokenizer("qwen_base")
instruct_model, instruct_tokenizer = load_model_and_tokenizer("qwen_instruct")

In [ ]:
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF-READ")

harmful_ds = load_dataset("walledai/AdvBench", token=hf_token)
harmless_ds = load_dataset("tatsu-lab/alpaca", token=hf_token)

harmful = harmful_ds["train"]
harmless = harmless_ds["train"]

harmful_prompts_raw = harmful["prompt"]

# Keep only Alpaca samples with empty input field
harmless_prompts_raw = [
    instruction
    for instruction, inp in zip(harmless["instruction"], harmless["input"])
    if inp.strip() == ""
]

n = min(len(harmful_prompts_raw), len(harmless_prompts_raw))

harmful_prompts_raw = harmful_prompts_raw[:n]
harmless_prompts_raw = harmless_prompts_raw[:n]

print(f"Harmful prompts  : {len(harmful_prompts_raw)}")
print(f"Harmless prompts : {len(harmless_prompts_raw)}")

In [ ]:
scores = torch.load("/kaggle/working/from-neurons-to-directions/data/scores.pt")

In [ ]:
k_values = [50, 100, 250, 500]
causal_effects = {}

for k in k_values:
    print(f"\n--- Evaluating k={k} neurons ---")
    top_k = get_top_safety_neurons(scores, k=k)
    C = compute_causal_effect(
        base_model=base_model,
        instruct_model=instruct_model,
        tokenizer=base_tokenizer,
        harmful_prompts=harmful_prompts_raw,
        safety_neurons=top_k,
        n_eval=10,       # increase to 20 for final results
    )
    causal_effects[k] = C

torch.save(causal_effects, "/kaggle/working/from-neurons-to-directions/causal_effects.pt")

fig = plot_causal_effect_bar(
    causal_effects=causal_effects,
    title="Causal Effect of Top-k Safety Neurons (Paper 2 Replication)",
    save_path="/kaggle/working/p2_causal_effect.png",
)
plt.show()